# 原始录制质量复算
使用 UMI `.venv` 的 Python kernel，先在 bash 中 source `/opt/ros/jazzy/setup.bash`。
下面是已用于外部 Python 脚本检查的同一份代码。本 notebook 未执行；运行可重新只读检查原始文件。
已完成的检查快照见 `raw-recording-check-20260930-160029.json` 与 `raw-recording-check-20260930-160029.md`。
范围：三个 episode 的 pose、TF、wrist_image、left_image；16.667/50 ms 是大间隔筛查阈值，并非硬件丢帧的直接证明。


In [ ]:
"""Read-only raw pose/video quality checks with exact timestamp arithmetic."""
import concurrent.futures
import datetime
import json
import math
import statistics
import subprocess
import time
from collections import Counter
from decimal import Decimal
from pathlib import Path

import numpy as np
import rosbag2_py
import yaml
from geometry_msgs.msg import PoseStamped
from rclpy.serialization import deserialize_message
from tf2_msgs.msg import TFMessage


SESSION = Path('/home/xing/Documents/UMI/recordings/'
               'online_session_20260930_160029_092182')
EXPECTED_POSE_HZ = 90.0


def stamp_ns(stamp):
    return stamp.sec * 10**9 + stamp.nanosec


def interval_stats(stamps, expected_hz):
    gaps = np.diff(np.asarray(stamps, dtype=np.int64))
    threshold_ns = 1.5 * 1e9 / expected_hz
    anomalous = np.flatnonzero(gaps > threshold_ns)
    return {
        'count': len(stamps),
        'duration_seconds': (stamps[-1] - stamps[0]) / 1e9,
        'average_hz': (len(stamps) - 1) * 1e9 / (stamps[-1] - stamps[0]),
        'median_gap_ms': float(np.median(gaps)) / 1e6,
        'p95_gap_ms': float(np.quantile(gaps, 0.95)) / 1e6,
        'p99_gap_ms': float(np.quantile(gaps, 0.99)) / 1e6,
        'min_gap_ms': int(gaps.min()) / 1e6,
        'max_gap_ms': int(gaps.max()) / 1e6,
        'non_increasing_timestamps': int(np.count_nonzero(gaps <= 0)),
        'large_gap_threshold_ms': threshold_ns / 1e6,
        'large_gap_count': len(anomalous),
        'gaps_over_100_ms': int(np.count_nonzero(gaps > 100_000_000)),
        'estimated_missing_periods_in_large_gaps': sum(
            max(0, round(int(gaps[i]) * expected_hz / 1e9) - 1)
            for i in anomalous),
        'large_gaps': [{
            'after_sample_index_zero_based': int(i),
            'at_seconds': (stamps[i] - stamps[0]) / 1e9,
            'gap_ms': int(gaps[i]) / 1e6,
            'previous_gap_ms': int(gaps[i - 1]) / 1e6 if i else None,
            'following_gap_ms': int(gaps[i + 1]) / 1e6
            if i + 1 < len(gaps) else None,
        } for i in anomalous],
    }


def pose_check(episode):
    reader = rosbag2_py.SequentialReader()
    reader.open(rosbag2_py.StorageOptions(uri=str(episode / 'pose'),
                                        storage_id='mcap'),
                rosbag2_py.ConverterOptions('', ''))
    headers, records, vectors, tf_headers, tf_records = [], [], [], [], []
    counts = Counter()
    frames = Counter()
    while reader.has_next():
        topic, data, recorded = reader.read_next()
        counts[topic] += 1
        if topic == '/vive/pose':
            message = deserialize_message(data, PoseStamped)
            headers.append(stamp_ns(message.header.stamp))
            records.append(recorded)
            frames[message.header.frame_id] += 1
            p, q = message.pose.position, message.pose.orientation
            vectors.append((p.x, p.y, p.z, q.x, q.y, q.z, q.w))
        elif topic == '/tf':
            message = deserialize_message(data, TFMessage)
            tf_headers.append(stamp_ns(message.transforms[0].header.stamp))
            tf_records.append(recorded)
    result = interval_stats(headers, EXPECTED_POSE_HZ)
    values = np.asarray(vectors)
    norm = np.linalg.norm(values[:, 3:], axis=1)
    lags_ms = (np.asarray(records, dtype=np.int64) -
               np.asarray(headers, dtype=np.int64)) / 1e6
    tf_lags_ms = (np.asarray(tf_records, dtype=np.int64) -
                  np.asarray(tf_headers, dtype=np.int64)) / 1e6
    metadata = yaml.safe_load((episode / 'pose' / 'metadata.yaml').read_text())
    metadata = metadata['rosbag2_bagfile_information']
    declared_counts = {i['topic_metadata']['name']: i['message_count']
                       for i in metadata['topics_with_message_count']}
    windows = []
    relative = (np.asarray(headers, dtype=np.int64) - headers[0]) / 1e9
    for start in range(0, int(relative[-1]) + 1, 5):
        selected = np.flatnonzero((relative >= start) & (relative < start + 5))
        if len(selected) < 2:
            continue
        stamps = [headers[i] for i in selected]
        windows.append({'start_seconds': start, 'samples': len(stamps),
                        'average_hz': (len(stamps) - 1) * 1e9 /
                                      (stamps[-1] - stamps[0])})
    result.update({
        'first_header_unix_ns': headers[0],
        'last_header_unix_ns': headers[-1],
        'first_record_unix_ns': records[0],
        'last_record_unix_ns': records[-1],
        'pose_record_timestamps_strictly_increasing': bool(np.all(
            np.diff(np.asarray(records, dtype=np.int64)) > 0)),
        'topic_counts': dict(counts),
        'metadata_counts_match': all(counts[t] == c for t, c in
                                     declared_counts.items()),
        'header_frames': dict(frames),
        'all_pose_components_finite': bool(np.isfinite(values).all()),
        'quaternion_norm_min': float(norm.min()),
        'quaternion_norm_max': float(norm.max()),
        'pose_receive_lag_median_ms': float(np.median(lags_ms)),
        'pose_receive_lag_p95_ms': float(np.quantile(lags_ms, 0.95)),
        'pose_receive_lag_max_ms': float(lags_ms.max()),
        'tf_receive_lag_median_ms': float(np.median(tf_lags_ms)),
        'matched_pose_tf_stamp_count': len(set(headers) & set(tf_headers)),
        'five_second_windows': windows,
    })
    return result, headers


def video_check(episode, stream, pose_result, pose_headers):
    video = episode / stream['video_path']
    timing = json.loads((episode / stream['timing_path']).read_text())
    prior = json.loads((episode / stream['validation_path']).read_text())
    probe = subprocess.run([
        'ffprobe', '-v', 'error', '-select_streams', 'v:0',
        '-show_entries', 'stream=codec_name,width,height,r_frame_rate,'
        'avg_frame_rate,duration,nb_frames:frame=best_effort_timestamp_time',
        '-of', 'json', str(video)], capture_output=True, text=True, check=True)
    media = json.loads(probe.stdout)
    frames = media['frames']
    pts_ns = [int(Decimal(f['best_effort_timestamp_time']) * 10**9)
              for f in frames if 'best_effort_timestamp_time' in f]
    assert len(pts_ns) == len(frames), f'Missing PTS in {video}'
    assert len(pts_ns) >= 2, f'Not enough frames in {video}'
    result = interval_stats(pts_ns, 30.0)
    image_times = np.asarray(pts_ns, dtype=np.int64) + timing[
        'first_receive_unix_ns']
    pose_times = np.asarray(pose_headers, dtype=np.int64)
    covered = (image_times >= pose_times[0]) & (image_times <= pose_times[-1])
    right = np.searchsorted(pose_times, image_times, side='right')
    valid = (right > 0) & (right < len(pose_times))
    bracket_gaps = pose_times[right[valid]] - pose_times[right[valid] - 1]
    source_span_ns = timing['last_source_pts_ns'] - timing['first_source_pts_ns']
    identity_matches = timing['camera_id'] == stream['camera_id'] and (
        timing['image_field'] == stream['image_field'])
    result.update({
        'camera_id': stream['camera_id'],
        'image_field': stream['image_field'],
        'relative_video_path': str(video.relative_to(SESSION)),
        'stream': media['streams'][0],
        'decoded_frames': len(frames),
        'accepted_frames': timing['accepted_frames'],
        'counts_match': len(frames) == timing['accepted_frames'],
        'probe_errors': probe.stderr.strip(),
        'all_frames_have_pts': len(pts_ns) == len(frames),
        'timing_status': timing['status'],
        'identity_matches_index': identity_matches,
        'timing_self_path_matches': timing['timing_path'] == str(
            episode / stream['timing_path']),
        'first_frame_host_unix_ns': int(image_times[0]),
        'last_frame_host_unix_ns': int(image_times[-1]),
        'frames_covered_by_pose_header_range': int(covered.sum()),
        'all_frames_covered': bool(covered.all()),
        'pose_lead_from_header_ms': (int(image_times[0]) -
                                     pose_times[0]) / 1e6,
        'pose_lead_from_record_ms': (int(image_times[0]) -
                                     pose_result['first_record_unix_ns']) / 1e6,
        'pose_tail_ms': (pose_times[-1] - int(image_times[-1])) / 1e6,
        'max_bracketing_pose_gap_ms': int(bracket_gaps.max()) / 1e6,
        'source_span_seconds': source_span_ns / 1e9,
        'pts_span_vs_source_span_difference_ms': ((pts_ns[-1] - pts_ns[0]) -
                                                 source_span_ns) / 1e6,
        'existing_validation_matches': (
            prior['decoded_frames'] == len(frames) and prior['decode_ok'] and
            prior['frame_count_matches']),
        'source_completeness': prior.get('source_completeness'),
    })
    for key in ('pose_lead_from_header_ms', 'pose_tail_ms'):
        result[key] = float(result[key])
    return result


def run_checks():
    started = time.perf_counter()
    index = yaml.safe_load((SESSION / 'session.yaml').read_text())
    episodes = [r for r in index['recordings'] if r['kind'] == 'episode']
    assert len(episodes) == 3, f'Expected three episodes, got {len(episodes)}'
    results, jobs = [], []
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        for recording in episodes:
            episode = SESSION / recording['path']
            pose_result, pose_headers = pose_check(episode)
            result = {'episode': episode.name,
                      'indexed_status': recording['status'],
                      'pose': pose_result, 'images': []}
            for stream in recording['image_streams']:
                jobs.append((result, pool.submit(video_check, episode, stream,
                                                  pose_result, pose_headers)))
            results.append(result)
        for result, future in jobs:
            result['images'].append(future.result())
    return {
        'session_id': index['session_id'],
        'session_status': index['status'],
        'task': index['task'],
        'session_started_at': index['started_at'],
        'session_ended_at': index['ended_at'],
        'temporal_calibration': index['temporal_calibration'],
        'checked_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'method': {
            'pose_gap_threshold': '1.5 / 90 seconds (16.667 ms)',
            'image_gap_threshold': '1.5 / 30 seconds (50 ms)',
            'coverage': 'Each decoded MP4 PTS anchored at first_receive_unix_ns '
                        'must lie between the first/last pose header stamp.',
            'limitations': 'Host receipt/source PTS are not exposure timestamps. '
                           'No per-camera calibrated offsets. Gap counts and '
                           'missing-period estimates cannot prove hardware '
                           'loss or locate it in Tracker/libsurvive/ROS/storage. '
                           'MP4 count equality verifies accepted-to-decoded '
                           'frames only; native per-frame camera sequence '
                           'numbers were not stored.',
        },
        'episodes': results,
        'inspection_elapsed_seconds': time.perf_counter() - started,
    }




In [ ]:
report = run_checks()
print(json.dumps(report, indent=2))
